# Train ModernBERT for span classification (PII-style spans)

This notebook fine-tunes a **ModernBERT encoder** for **span classification**:

- encode each document once
- enumerate candidate spans up to `max_span_len`
- classify each span as `NONE` or a concrete PII label

It is designed as a practical baseline for containerized or local runs with Hugging Face Transformers + PyTorch.

## Source of truth for labels

Label taxonomy is imported from `pii_labels.py` via `TRAINING_LABELS`.

- `LABELS = TRAINING_LABELS`
- expected shape is `['NONE', ...PII labels...]`
- this keeps training labels aligned with data generation and avoids drift

If `pii_labels.py` is unavailable, the notebook falls back to an embedded label list so the notebook still runs.

## Accepted input JSONL formats

Each line must include `text` and may provide supervision in either of two formats.

### Format A: direct spans (already normalized)

```json
{"text": "John Smith works at Google in Zurich.", "spans": [{"start": 0, "end": 10, "label": "PERSON"}, {"start": 20, "end": 26, "label": "ORG"}]}
```

### Format B: generator-style items (category-aware)

```json
{"text": "...", "items": [{"start": 10, "end": 18, "category": "NON_PII_NUMBER", "note": "invoice amount"}, {"start": 50, "end": 66, "category": "REAL_PII", "label": "PHONE"}]}
```

Rows in Format B are normalized in `load_jsonl()` by converting:

- `category == 'REAL_PII'` -> training `spans` (uses `label`)
- `category == 'NON_PII_NUMBER'` -> not converted to labeled spans
- `category == 'PII_LOOKALIKE'` -> not converted to labeled spans

The loader prints a conversion message when `items -> spans` normalization happens.

## How non-PII and lookalike examples are used

`NON_PII_NUMBER` and `PII_LOOKALIKE` content is still important even though it is not converted to labeled spans:

- it remains in the source text
- candidate spans are enumerated across that text
- unlabeled candidate spans become `NONE`

This means those hard negatives still train the classifier boundary and type discrimination behavior.

## Span semantics

For all labeled spans:

- `start` is **inclusive**
- `end` is **exclusive**
- offsets are measured on raw `text`

At runtime, character spans are mapped to token spans with `char_span_to_token_span()`.

## Modeling notes

The span representation is intentionally simple:

- start token embedding
- end token embedding
- learned span-length embedding
- optional mean pool over span interior

This is meant to stay compact and debuggable while producing a strong baseline.


In [74]:
# Uncomment and run if you need installs.
# For Windows + CUDA 12.8, install PyTorch from the official selector for your setup.
# Example pattern (version may change):
# !pip install --upgrade pip
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
# !pip install -U transformers datasets accelerate evaluate scikit-learn seqeval tqdm

import gc
import json
import math
import os
import random
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoConfig, AutoModel, get_linear_schedule_with_warmup


In [75]:
# -----------------------------
# Configuration
# -----------------------------
MODEL_NAME = "answerdotai/ModernBERT-base"   # swap to -large if you enjoy VRAM bonfires
# Use all vendor/model datasets by default (e.g., train.gpt-5-mini.json, train.gpt-5-nano.json)
TRAIN_PATH = "train.*.json*"
VALID_PATH = "valid.*.json*"

OUTPUT_DIR = "./modernbert_span_model"
MAX_LENGTH = 512
MAX_SPAN_LEN = 15
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 300
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 10
MIN_EPOCHS = 40
MIN_DELTA = 1e-5
LENGTH_EMBED_DIM = 32
DROPOUT = 0.1

# Checkpoint / logging controls
SAVE_EVERY_N_EPOCHS = 10
LOG_FULL_METRICS_EVERY_N_EPOCHS = 5
ENABLE_MEMORY_HOUSEKEEPING = True
LOG_MEMORY_EVERY_EPOCH = True

# Resume options (useful after OOM/crash).
RESUME_TRAINING = False
RESUME_CHECKPOINT_PATH = None

# Loss mix: CE keeps optimization stable. Enable Soft-F1 only after baseline learns positives.
USE_SOFT_F1_LOSS = False
SOFT_F1_WEIGHT = 0.2
SOFT_F1_EXCLUDE_NONE = True

# Heavy NONE imbalance can collapse training to all-NONE predictions.
# This is used as a fallback default and will be overridden by the epoch schedule below.
NEGATIVE_SPAN_SUBSAMPLE = 0.25
NONE_START_PREVALENCE = 0.10
NONE_FULL_PREVALENCE_EPOCH = 60

# Curriculum / span sampling controls.
# Start with exact positives + clear NONE. Introduce partial-overlap negatives later.
PARTIAL_OVERLAP_START_EPOCH = 12
PARTIAL_OVERLAP_FULL_EPOCH = 36
PARTIAL_OVERLAP_MAX_WEIGHT = 0.20
HARD_NEGATIVES_PER_POS = 2
EASY_NEGATIVES_PER_POS = 6
MAX_NEGATIVES_NO_POS = 48

# Decode overlap suppression (NMS-like).
DECODE_SAME_LABEL_OVERLAP_THRESHOLD = 0.35
DECODE_CROSS_LABEL_OVERLAP_THRESHOLD = 0.70
DECODE_ENABLE_CROSS_LABEL_NMS = True

# Per-label threshold calibration (validation-driven).
USE_PER_LABEL_THRESHOLDING = True
PER_LABEL_THRESHOLD_WARMUP_EPOCHS = 8
PER_LABEL_CALIBRATION_EVERY_N_EPOCHS = 2
PER_LABEL_CALIBRATION_MIN_SUPPORT = 40
LOG_CALIBRATION_TOP_K = 8
LOG_DEAD_LABELS_MIN_SUPPORT = 20
LOG_REACHABILITY_SUMMARY = True

# Keep threshold explicit for monitoring/checkpointing.
# Used as fallback for labels that are not calibrated.
EVAL_THRESHOLD = 0.20
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("device:", DEVICE)


device: cuda


In [76]:
# -----------------------------
# Label set
# -----------------------------
# Keep these aligned with labels present in your JSONL spans.
# Primary path: import shared taxonomy used by data generation.
try:
    from pii_labels import TRAINING_LABELS
    LABELS = list(TRAINING_LABELS)
except Exception:
    # Fallback keeps the notebook runnable even if the module is unavailable.
    LABELS = [
        "NONE",
        "PERSON",
        "ORG",
        "ADDRESS",
        "EMAIL",
        "PHONE",
        "USERNAME",
        "PASSWORD",
        "IP_ADDRESS",
        "IBAN",
        "CREDIT_CARD",
        "ID_NUMBER",
        "ACCOUNT_NUMBER",
        "OTHER",
    ]

ENTITY_LABELS = [label for label in LABELS if label != "NONE"]
label2id = {x: i for i, x in enumerate(LABELS)}
id2label = {i: x for x, i in label2id.items()}
NUM_LABELS = len(LABELS)

label2id


{'NONE': 0,
 'PERSON': 1,
 'ORG': 2,
 'ADDRESS': 3,
 'EMAIL': 4,
 'PHONE': 5,
 'USERNAME': 6,
 'PASSWORD': 7,
 'IP_ADDRESS': 8,
 'IBAN': 9,
 'CREDIT_CARD': 10,
 'ID_NUMBER': 11,
 'ACCOUNT_NUMBER': 12,
 'OTHER': 13}

In [ ]:
# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer


TokenizersBackend(name_or_path='answerdotai/ModernBERT-base', vocab_size=50280, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("|||IP_ADDRESS|||", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	1: AddedToken("<|padding|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50254: AddedToken("                        ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50255: AddedToken("                       ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50256: AddedToken("                      ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50257: AddedToken("                     ", rstrip=False, lstrip=False, single_word=False, norm

In [78]:
# -----------------------------
# Data helpers
# -----------------------------
def normalize_row(row: dict) -> dict:
    text = row.get("text", "")
    if not isinstance(text, str):
        raise ValueError("Each row must include a string 'text' field.")

    if isinstance(row.get("spans"), list):
        spans = row["spans"]
        return {"text": text, "spans": spans}

    # Backward/alternate format support:
    # if rows contain "items", keep only REAL_PII entries as trainable labeled spans.
    items = row.get("items", [])
    spans = []
    if isinstance(items, list):
        for it in items:
            if not isinstance(it, dict):
                continue
            if it.get("category") != "REAL_PII":
                continue
            start = it.get("start")
            end = it.get("end")
            label = it.get("label")
            if isinstance(start, int) and isinstance(end, int) and isinstance(label, str):
                spans.append({"start": start, "end": end, "label": label})

    return {"text": text, "spans": spans}


def load_jsonl(path: str) -> List[dict]:
    import glob

    matched_paths = sorted(glob.glob(path))
    if not matched_paths and os.path.exists(path):
        matched_paths = [path]

    if not matched_paths:
        raise FileNotFoundError(f"No files matched: {path}")

    rows = []
    converted_from_items = 0
    for matched in matched_paths:
        with open(matched, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                raw = json.loads(line)
                if isinstance(raw.get("items"), list) and not isinstance(raw.get("spans"), list):
                    converted_from_items += 1
                rows.append(normalize_row(raw))

    if len(matched_paths) > 1:
        print(f"[load_jsonl] Loaded {len(matched_paths)} files for pattern: {path}")
    if converted_from_items:
        print(f"[load_jsonl] Converted {converted_from_items} rows from items->spans in {path}")
    return rows


def char_span_to_token_span(offset_mapping: List[Tuple[int, int]], start_char: int, end_char: int) -> Optional[Tuple[int, int]]:
    """
    Convert [start_char, end_char) to token span [start_tok, end_tok], inclusive.
    Only tokens with non-zero-length offsets are considered.
    """
    token_start = None
    token_end = None

    for i, (s, e) in enumerate(offset_mapping):
        if s == e:
            continue
        if token_start is None and s <= start_char < e:
            token_start = i
        # token fully or partially touching the end-1 char
        if s < end_char <= e:
            token_end = i

    # fallback for spans aligning on token boundaries
    if token_start is None:
        for i, (s, e) in enumerate(offset_mapping):
            if s == e:
                continue
            if s >= start_char:
                token_start = i
                break

    if token_end is None:
        for i in range(len(offset_mapping) - 1, -1, -1):
            s, e = offset_mapping[i]
            if s == e:
                continue
            if e <= end_char:
                token_end = i
                break

    if token_start is None or token_end is None or token_start > token_end:
        return None
    return token_start, token_end


def enumerate_spans(num_tokens: int, max_span_len: int) -> List[Tuple[int, int]]:
    spans = []
    for i in range(num_tokens):
        for j in range(i, min(i + max_span_len, num_tokens)):
            spans.append((i, j))
    return spans


In [79]:
class SpanDataset(Dataset):
    def __init__(
        self,
        path: str,
        tokenizer,
        label2id: Dict[str, int],
        max_length: int = 512,
        max_span_len: int = 6,
        negative_span_subsample: float = 1.0,
        training: bool = True,
        hard_negatives_per_pos: int = 4,
        easy_negatives_per_pos: int = 12,
        max_negatives_no_pos: int = 64,
        partial_overlap_weight: float = 0.0,
    ):
        self.rows = load_jsonl(path)
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        self.max_span_len = max_span_len
        self.negative_span_subsample = negative_span_subsample
        self.training = training
        self.hard_negatives_per_pos = hard_negatives_per_pos
        self.easy_negatives_per_pos = easy_negatives_per_pos
        self.max_negatives_no_pos = max_negatives_no_pos
        self.partial_overlap_weight = partial_overlap_weight

    def __len__(self):
        return len(self.rows)

    @staticmethod
    def _has_token_overlap(span_a: Tuple[int, int], span_b: Tuple[int, int]) -> bool:
        a0, a1 = span_a
        b0, b1 = span_b
        return not (a1 < b0 or b1 < a0)

    def __getitem__(self, idx):
        row = self.rows[idx]
        text = row["text"]
        gold_spans = row.get("spans", [])

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_attention_mask=True,
        )

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]
        offset_mapping = enc["offset_mapping"]

        valid_token_indices = [i for i, (s, e) in enumerate(offset_mapping) if e > s]
        span_label_map = {}
        gold_token_spans = []

        for gs in gold_spans:
            mapped = char_span_to_token_span(offset_mapping, gs["start"], gs["end"])
            if mapped is None:
                continue

            raw_label = gs["label"]
            if raw_label not in self.label2id:
                raise ValueError(
                    f"Unknown span label '{raw_label}' in dataset row {idx}. "
                    f"Expected one of: {sorted(self.label2id.keys())}. "
                    "Update ENTITY_LABELS or normalize your dataset labels."
                )

            label_id = self.label2id[raw_label]
            span_label_map[mapped] = label_id
            gold_token_spans.append((mapped[0], mapped[1], label_id))

        positives = []
        overlap_negatives = []
        clear_negatives = []
        none_id = self.label2id["NONE"]

        # only enumerate on tokens that correspond to real text
        # this avoids special tokens with zero-length offsets
        real_positions = valid_token_indices

        for start_rank in range(len(real_positions)):
            for end_rank in range(start_rank, min(start_rank + self.max_span_len, len(real_positions))):
                start_tok = real_positions[start_rank]
                end_tok = real_positions[end_rank]
                span_key = (start_tok, end_tok)

                exact_label = span_label_map.get(span_key)
                if exact_label is not None:
                    positives.append((start_tok, end_tok, exact_label, 1.0))
                    continue

                overlaps_gold = any(
                    self._has_token_overlap((start_tok, end_tok), (gs, ge))
                    for gs, ge, _ in gold_token_spans
                )

                neg_item = (start_tok, end_tok, none_id)
                if overlaps_gold:
                    overlap_negatives.append(neg_item)
                else:
                    clear_negatives.append(neg_item)

        selected = []
        if self.training:
            num_pos = len(positives)
            selected.extend(positives)

            if num_pos > 0:
                hard_budget = max(1, self.hard_negatives_per_pos * num_pos)
                easy_budget = max(1, self.easy_negatives_per_pos * num_pos)
            else:
                hard_budget = 0
                easy_budget = self.max_negatives_no_pos

            if self.partial_overlap_weight > 0.0 and overlap_negatives and hard_budget > 0:
                hard_take = random.sample(overlap_negatives, min(len(overlap_negatives), hard_budget))
                selected.extend((s, e, lbl, self.partial_overlap_weight) for s, e, lbl in hard_take)

            clear_pool = clear_negatives
            if self.negative_span_subsample < 1.0:
                clear_pool = [
                    x for x in clear_pool
                    if random.random() <= self.negative_span_subsample
                ]

            if clear_pool and easy_budget > 0:
                easy_take = random.sample(clear_pool, min(len(clear_pool), easy_budget))
                selected.extend((s, e, lbl, 1.0) for s, e, lbl in easy_take)

            if not selected:
                fallback_pool = clear_negatives or overlap_negatives
                if fallback_pool:
                    s, e, lbl = random.choice(fallback_pool)
                    selected.append((s, e, lbl, 1.0))
        else:
            selected.extend(positives)
            selected.extend((s, e, lbl, 1.0) for s, e, lbl in overlap_negatives)
            selected.extend((s, e, lbl, 1.0) for s, e, lbl in clear_negatives)

        candidate_spans = [(s, e) for s, e, _, _ in selected]
        candidate_labels = [lbl for _, _, lbl, _ in selected]
        loss_weights = [w for _, _, _, w in selected]

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "spans": torch.tensor(candidate_spans, dtype=torch.long) if candidate_spans else torch.zeros((0, 2), dtype=torch.long),
            "labels": torch.tensor(candidate_labels, dtype=torch.long) if candidate_labels else torch.zeros((0,), dtype=torch.long),
            "loss_weights": torch.tensor(loss_weights, dtype=torch.float32) if loss_weights else torch.zeros((0,), dtype=torch.float32),
            "text": text,
            "offset_mapping": offset_mapping,
            "gold_spans": gold_spans,
        }
        return item


In [80]:
@dataclass
class Batch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    spans: torch.Tensor
    span_batch_ix: torch.Tensor
    labels: torch.Tensor
    loss_weights: torch.Tensor
    meta: List[dict]

def collate_fn(features: List[dict]) -> Batch:
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    spans = []
    span_batch_ix = []
    labels = []
    loss_weights = []
    meta = []

    for b_ix, f in enumerate(features):
        pad_len = max_len - len(f["input_ids"])
        input_ids.append(F.pad(f["input_ids"], (0, pad_len), value=tokenizer.pad_token_id))
        attention_mask.append(F.pad(f["attention_mask"], (0, pad_len), value=0))

        if len(f["spans"]) > 0:
            spans.append(f["spans"])
            span_batch_ix.append(torch.full((len(f["spans"]),), b_ix, dtype=torch.long))
            labels.append(f["labels"])
            loss_weights.append(f["loss_weights"])

        meta.append({
            "text": f["text"],
            "offset_mapping": f["offset_mapping"],
            "gold_spans": f["gold_spans"],
        })

    if spans:
        spans = torch.cat(spans, dim=0)
        span_batch_ix = torch.cat(span_batch_ix, dim=0)
        labels = torch.cat(labels, dim=0)
        loss_weights = torch.cat(loss_weights, dim=0)
    else:
        spans = torch.zeros((0, 2), dtype=torch.long)
        span_batch_ix = torch.zeros((0,), dtype=torch.long)
        labels = torch.zeros((0,), dtype=torch.long)
        loss_weights = torch.zeros((0,), dtype=torch.float32)

    return Batch(
        input_ids=torch.stack(input_ids),
        attention_mask=torch.stack(attention_mask),
        spans=spans,
        span_batch_ix=span_batch_ix,
        labels=labels,
        loss_weights=loss_weights,
        meta=meta,
    )


In [81]:
# -----------------------------
# Model
# -----------------------------
def soft_f1_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    num_labels: int,
    exclude_none: bool = True,
    weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Differentiable macro Soft-F1 over classes.
    This is a surrogate objective (not exact span-set F1), used to better align
    training with F1-based model selection.
    """
    probs = torch.softmax(logits, dim=-1)  # [S, C]
    target = F.one_hot(labels, num_classes=num_labels).float()  # [S, C]

    start_idx = 1 if exclude_none else 0
    probs = probs[:, start_idx:]
    target = target[:, start_idx:]

    if weights is not None:
        w = weights.float().unsqueeze(1)
        probs = probs * w
        target = target * w

    tp = (probs * target).sum(dim=0)
    fp = (probs * (1.0 - target)).sum(dim=0)
    fn = ((1.0 - probs) * target).sum(dim=0)

    soft_f1_per_class = (2.0 * tp + 1e-9) / (2.0 * tp + fp + fn + 1e-9)
    return 1.0 - soft_f1_per_class.mean()


class ModernBertForSpanClassification(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_labels: int,
        max_span_len: int = 6,
        length_embed_dim: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.length_emb = nn.Embedding(max_span_len + 1, length_embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 3 + length_embed_dim, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_labels),
        )
        self.max_span_len = max_span_len
        self.num_labels = num_labels

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        spans: torch.Tensor,
        span_batch_ix: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        loss_weights: Optional[torch.Tensor] = None,
    ):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state  # [B, T, H]

        if spans.size(0) == 0:
            logits = torch.zeros((0, self.num_labels), device=h.device)
            loss = None
            if labels is not None:
                loss = logits.sum() * 0.0
            return {"loss": loss, "logits": logits}

        start_ix = spans[:, 0]
        end_ix = spans[:, 1]

        start_h = h[span_batch_ix, start_ix]  # [S, H]
        end_h = h[span_batch_ix, end_ix]      # [S, H]

        # Mean pool inside each span. Simple and decent.
        pooled = []
        for k in range(spans.size(0)):
            b = span_batch_ix[k]
            s = start_ix[k].item()
            e = end_ix[k].item()
            pooled.append(h[b, s:e+1].mean(dim=0))
        pooled_h = torch.stack(pooled, dim=0)  # [S, H]

        span_len = (end_ix - start_ix + 1).clamp(max=self.max_span_len)
        len_h = self.length_emb(span_len)

        span_repr = torch.cat([start_h, end_h, pooled_h, len_h], dim=-1)
        span_repr = self.dropout(span_repr)
        logits = self.classifier(span_repr)

        loss = None
        if labels is not None:
            ce_raw = F.cross_entropy(logits, labels, reduction="none")
            if loss_weights is not None:
                weights = loss_weights.float().to(logits.device)
                denom = weights.sum().clamp_min(1e-9)
                ce_loss = (ce_raw * weights).sum() / denom
            else:
                ce_loss = ce_raw.mean()

            if USE_SOFT_F1_LOSS:
                f1_surrogate = soft_f1_loss(
                    logits,
                    labels,
                    num_labels=self.num_labels,
                    exclude_none=SOFT_F1_EXCLUDE_NONE,
                    weights=loss_weights,
                )
                loss = ce_loss + SOFT_F1_WEIGHT * f1_surrogate
            else:
                loss = ce_loss

        return {"loss": loss, "logits": logits}


In [82]:
# -----------------------------
# Create datasets / loaders
# -----------------------------
train_ds = SpanDataset(
    TRAIN_PATH,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
    max_span_len=MAX_SPAN_LEN,
    negative_span_subsample=NEGATIVE_SPAN_SUBSAMPLE,
    training=True,
    hard_negatives_per_pos=HARD_NEGATIVES_PER_POS,
    easy_negatives_per_pos=EASY_NEGATIVES_PER_POS,
    max_negatives_no_pos=MAX_NEGATIVES_NO_POS,
    partial_overlap_weight=0.0,
)

valid_ds = SpanDataset(
    VALID_PATH,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
    max_span_len=MAX_SPAN_LEN,
    negative_span_subsample=1.0,
    training=False,
    hard_negatives_per_pos=HARD_NEGATIVES_PER_POS,
    easy_negatives_per_pos=EASY_NEGATIVES_PER_POS,
    max_negatives_no_pos=MAX_NEGATIVES_NO_POS,
    partial_overlap_weight=1.0,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_ds), len(valid_ds)


(1425, 290)

In [83]:
# -----------------------------
# Build model / optimizer / scheduler
# -----------------------------
model = ModernBertForSpanClassification(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    max_span_len=MAX_SPAN_LEN,
    length_embed_dim=LENGTH_EMBED_DIM,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
num_training_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

num_training_steps, num_warmup_steps


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 10906.98it/s]
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(27000, 2700)

In [84]:
# -----------------------------
# Metrics
# -----------------------------
def spans_to_typed_set(pred_spans: List[Tuple[int, int, str]]) -> set:
    return set(pred_spans)


def _token_overlap_ratio_min_len(span_a: Tuple[int, int], span_b: Tuple[int, int]) -> float:
    a0, a1 = span_a
    b0, b1 = span_b
    inter = max(0, min(a1, b1) - max(a0, b0) + 1)
    if inter <= 0:
        return 0.0
    len_a = a1 - a0 + 1
    len_b = b1 - b0 + 1
    return inter / float(min(len_a, len_b))


def _nms_by_label(preds: List[Tuple[int, int, str, float]], overlap_threshold: float) -> List[Tuple[int, int, str, float]]:
    if not preds:
        return []

    preds = sorted(preds, key=lambda x: x[3], reverse=True)
    kept = []
    for cand in preds:
        cs, ce, _, _ = cand
        suppress = any(
            _token_overlap_ratio_min_len((cs, ce), (ks, ke)) > overlap_threshold
            for ks, ke, _, _ in kept
        )
        if not suppress:
            kept.append(cand)
    return kept


def _cross_label_nms(preds: List[Tuple[int, int, str, float]], overlap_threshold: float) -> List[Tuple[int, int, str, float]]:
    if not preds:
        return []

    preds = sorted(preds, key=lambda x: x[3], reverse=True)
    kept = []
    for cand in preds:
        cs, ce, _, _ = cand
        suppress = any(
            _token_overlap_ratio_min_len((cs, ce), (ks, ke)) > overlap_threshold
            for ks, ke, _, _ in kept
        )
        if not suppress:
            kept.append(cand)
    return kept


def decode_predictions(
    batch: Batch,
    logits: torch.Tensor,
    threshold: float = 0.5,
    label_thresholds: Optional[Dict[str, float]] = None,
):
    """
    Decode span predictions per item.
    Per-label NMS keeps the highest-scoring overlap cluster member.
    Optional cross-label suppression then resolves conflicting labels on the same region.
    """
    probs = torch.softmax(logits, dim=-1)
    pred_label_ids = probs.argmax(dim=-1)
    pred_scores = probs.max(dim=-1).values

    by_item = [[] for _ in range(len(batch.meta))]
    for i in range(len(batch.spans)):
        b = batch.span_batch_ix[i].item()
        s, e = batch.spans[i].tolist()
        lbl_id = pred_label_ids[i].item()
        score = pred_scores[i].item()

        if lbl_id == label2id["NONE"]:
            continue

        lbl_name = id2label[lbl_id]
        effective_threshold = threshold
        if label_thresholds is not None:
            effective_threshold = label_thresholds.get(lbl_name, threshold)

        if score >= effective_threshold:
            by_item[b].append((s, e, lbl_name, score))

    resolved = []
    for item_preds in by_item:
        by_label: Dict[str, List[Tuple[int, int, str, float]]] = {}
        for pred in item_preds:
            by_label.setdefault(pred[2], []).append(pred)

        kept = []
        for _, label_preds in by_label.items():
            kept.extend(_nms_by_label(label_preds, DECODE_SAME_LABEL_OVERLAP_THRESHOLD))

        if DECODE_ENABLE_CROSS_LABEL_NMS:
            kept = _cross_label_nms(kept, DECODE_CROSS_LABEL_OVERLAP_THRESHOLD)

        kept.sort(key=lambda x: x[3], reverse=True)
        resolved.append(kept)
    return resolved


def token_spans_to_char_spans(preds, offset_mapping):
    out = []
    for s, e, lbl, score in preds:
        start_char = offset_mapping[s][0]
        end_char = offset_mapping[e][1]
        out.append((start_char, end_char, lbl, score))
    return out


def _compute_metrics_from_sets(
    all_gold: List[Tuple[int, int, int, str]],
    all_pred: List[Tuple[int, int, int, str]],
    total_scored_spans: int,
    gold_label_counts: Counter,
    pred_argmax_label_counts: Counter,
    pred_decoded_label_counts: Counter,
    wrong_place_count: int,
    wrong_place_label_counts: Counter,
    wrong_place_overlap_count: int,
    wrong_place_overlap_label_counts: Counter,
):
    gold_set = set(all_gold)
    pred_set = set(all_pred)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    ordered_labels = [lbl for lbl in LABELS if lbl != "NONE"]

    gold_by_label = {
        lbl: {item for item in gold_set if item[3] == lbl}
        for lbl in ordered_labels
    }
    pred_by_label = {
        lbl: {item for item in pred_set if item[3] == lbl}
        for lbl in ordered_labels
    }

    per_label_metrics = {}
    for lbl in ordered_labels:
        lbl_tp = len(gold_by_label[lbl] & pred_by_label[lbl])
        lbl_fp = len(pred_by_label[lbl] - gold_by_label[lbl])
        lbl_fn = len(gold_by_label[lbl] - pred_by_label[lbl])

        lbl_precision = lbl_tp / (lbl_tp + lbl_fp + 1e-9)
        lbl_recall = lbl_tp / (lbl_tp + lbl_fn + 1e-9)
        lbl_f1 = 2 * lbl_precision * lbl_recall / (lbl_precision + lbl_recall + 1e-9)

        per_label_metrics[lbl] = {
            "support": int(len(gold_by_label[lbl])),
            "tp": int(lbl_tp),
            "fp": int(lbl_fp),
            "fn": int(lbl_fn),
            "precision": lbl_precision,
            "recall": lbl_recall,
            "f1": lbl_f1,
        }

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "num_scored_spans": total_scored_spans,
        "gold_label_counts": {lbl: int(gold_label_counts.get(lbl, 0)) for lbl in ordered_labels},
        "pred_argmax_label_counts": {lbl: int(pred_argmax_label_counts.get(lbl, 0)) for lbl in LABELS},
        "pred_decoded_label_counts": {lbl: int(pred_decoded_label_counts.get(lbl, 0)) for lbl in ordered_labels},
        "wrong_place_count": int(wrong_place_count),
        "wrong_place_label_counts": {lbl: int(wrong_place_label_counts.get(lbl, 0)) for lbl in ordered_labels},
        "wrong_place_overlap_count": int(wrong_place_overlap_count),
        "wrong_place_overlap_label_counts": {lbl: int(wrong_place_overlap_label_counts.get(lbl, 0)) for lbl in ordered_labels},
        "per_label_metrics": per_label_metrics,
    }


def fit_label_thresholds_from_argmax(
    model,
    loader,
    default_threshold: float = 0.5,
    min_support: int = 20,
):
    """
    Fit per-label thresholds from validation argmax scores in one pass.
    For each predicted argmax label, we model exact-match correctness vs score and
    choose the threshold maximizing that label's own F1 surrogate.
    Returns (thresholds, calibration_report).
    """
    model.eval()
    labels_wo_none = [lbl for lbl in LABELS if lbl != "NONE"]
    label_examples: Dict[str, List[Tuple[float, int]]] = {lbl: [] for lbl in labels_wo_none}
    label_support: Dict[str, int] = {lbl: 0 for lbl in labels_wo_none}

    with torch.no_grad():
        for batch in tqdm(loader, desc="fit label thresholds"):
            input_ids = batch.input_ids.to(DEVICE)
            attention_mask = batch.attention_mask.to(DEVICE)
            spans = batch.spans.to(DEVICE)
            span_batch_ix = batch.span_batch_ix.to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                spans=spans,
                span_batch_ix=span_batch_ix,
                labels=None,
            )

            probs = torch.softmax(outputs["logits"].cpu(), dim=-1)
            pred_label_ids = probs.argmax(dim=-1)
            pred_scores = probs.max(dim=-1).values

            gold_by_item = []
            for meta in batch.meta:
                gold_set = {(gs["start"], gs["end"], gs["label"]) for gs in meta["gold_spans"]}
                gold_by_item.append(gold_set)
                for _, _, lbl in gold_set:
                    if lbl in label_support:
                        label_support[lbl] += 1

            for i in range(len(batch.spans)):
                item_ix = batch.span_batch_ix[i].item()
                s_tok, e_tok = batch.spans[i].tolist()
                lbl_id = int(pred_label_ids[i].item())
                if lbl_id == label2id["NONE"]:
                    continue

                lbl_name = id2label[lbl_id]
                if lbl_name not in label_examples:
                    continue

                s_char = batch.meta[item_ix]["offset_mapping"][s_tok][0]
                e_char = batch.meta[item_ix]["offset_mapping"][e_tok][1]
                is_tp = int((s_char, e_char, lbl_name) in gold_by_item[item_ix])
                score = float(pred_scores[i].item())
                label_examples[lbl_name].append((score, is_tp))

    thresholds = {lbl: float(default_threshold) for lbl in label_examples.keys()}
    report = {}

    for lbl, examples in label_examples.items():
        support = int(label_support.get(lbl, 0))
        num_examples = int(len(examples))
        default_used = support < min_support or num_examples == 0

        if default_used:
            report[lbl] = {
                "support": support,
                "examples": num_examples,
                "best_threshold": float(default_threshold),
                "best_f1_surrogate": 0.0,
                "default_used": True,
            }
            continue

        examples_sorted = sorted(examples, key=lambda x: x[0], reverse=True)
        tp = 0
        fp = 0
        best_f1 = -1.0
        best_thr = float(default_threshold)

        idx = 0
        while idx < len(examples_sorted):
            thr = examples_sorted[idx][0]
            while idx < len(examples_sorted) and examples_sorted[idx][0] == thr:
                if examples_sorted[idx][1] == 1:
                    tp += 1
                else:
                    fp += 1
                idx += 1

            fn = max(support - tp, 0)
            p = tp / (tp + fp + 1e-9)
            r = tp / (tp + fn + 1e-9)
            f1 = 2 * p * r / (p + r + 1e-9)

            if f1 > best_f1:
                best_f1 = f1
                best_thr = float(thr)

        best_thr = float(min(max(best_thr, 0.0), 1.0))
        thresholds[lbl] = best_thr
        report[lbl] = {
            "support": support,
            "examples": num_examples,
            "best_threshold": best_thr,
            "best_f1_surrogate": float(max(best_f1, 0.0)),
            "default_used": False,
        }

    return thresholds, report


def evaluate_model(
    model,
    loader,
    threshold: float = 0.5,
    label_thresholds: Optional[Dict[str, float]] = None,
):
    model.eval()
    all_gold = []
    all_pred = []

    gold_label_counts = Counter()
    pred_decoded_label_counts = Counter()
    pred_argmax_label_counts = Counter()
    wrong_place_label_counts = Counter()
    wrong_place_overlap_label_counts = Counter()

    total_scored_spans = 0
    wrong_place_count = 0
    wrong_place_overlap_count = 0
    item_id = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval"):
            input_ids = batch.input_ids.to(DEVICE)
            attention_mask = batch.attention_mask.to(DEVICE)
            spans = batch.spans.to(DEVICE)
            span_batch_ix = batch.span_batch_ix.to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                spans=spans,
                span_batch_ix=span_batch_ix,
                labels=None,
            )

            logits_cpu = outputs["logits"].cpu()
            argmax_ids = logits_cpu.argmax(dim=-1).tolist()
            total_scored_spans += len(argmax_ids)
            for lbl_id in argmax_ids:
                pred_argmax_label_counts[id2label[int(lbl_id)]] += 1

            resolved = decode_predictions(
                batch,
                logits_cpu,
                threshold=threshold,
                label_thresholds=label_thresholds,
            )

            for meta, preds in zip(batch.meta, resolved):
                pred_char = [(s, e, lbl) for s, e, lbl, score in token_spans_to_char_spans(preds, meta["offset_mapping"])]
                gold_char = [(gs["start"], gs["end"], gs["label"]) for gs in meta["gold_spans"]]

                for _, _, lbl in pred_char:
                    pred_decoded_label_counts[lbl] += 1
                for gs in meta["gold_spans"]:
                    gold_label_counts[gs["label"]] += 1

                gold_set_item = set(gold_char)
                gold_spans_by_label = {}
                for gs, ge, glbl in gold_char:
                    gold_spans_by_label.setdefault(glbl, []).append((gs, ge))

                for ps, pe, plbl in pred_char:
                    if (ps, pe, plbl) in gold_set_item:
                        continue
                    if plbl in gold_spans_by_label:
                        wrong_place_count += 1
                        wrong_place_label_counts[plbl] += 1

                        overlaps_same_label = any(
                            not (pe <= gs or ps >= ge)
                            for gs, ge in gold_spans_by_label[plbl]
                        )
                        if overlaps_same_label:
                            wrong_place_overlap_count += 1
                            wrong_place_overlap_label_counts[plbl] += 1

                all_pred.extend((item_id, s, e, lbl) for s, e, lbl in pred_char)
                all_gold.extend((item_id, s, e, lbl) for s, e, lbl in gold_char)
                item_id += 1

    metrics = _compute_metrics_from_sets(
        all_gold=all_gold,
        all_pred=all_pred,
        total_scored_spans=total_scored_spans,
        gold_label_counts=gold_label_counts,
        pred_argmax_label_counts=pred_argmax_label_counts,
        pred_decoded_label_counts=pred_decoded_label_counts,
        wrong_place_count=wrong_place_count,
        wrong_place_label_counts=wrong_place_label_counts,
        wrong_place_overlap_count=wrong_place_overlap_count,
        wrong_place_overlap_label_counts=wrong_place_overlap_label_counts,
    )
    if label_thresholds is not None:
        metrics["label_thresholds"] = {k: float(v) for k, v in label_thresholds.items()}
    return metrics


In [85]:
# -----------------------------
# Training loop
# -----------------------------
EARLY_STOPPING_PATIENCE = 10
MIN_EPOCHS = 30
MIN_DELTA = 1e-4

LOG_MEMORY_EVERY_EPOCH = globals().get("LOG_MEMORY_EVERY_EPOCH", True)

def _log_memory(epoch_num: int):
    sys_msg = "system_mem_used=unavailable"
    try:
        mem_total_kb = None
        mem_avail_kb = None
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    mem_total_kb = int(line.split()[1])
                elif line.startswith('MemAvailable:'):
                    mem_avail_kb = int(line.split()[1])
        if mem_total_kb is not None and mem_avail_kb is not None:
            mem_used_mb = (mem_total_kb - mem_avail_kb) / 1024.0
            mem_total_mb = mem_total_kb / 1024.0
            sys_msg = f"system_mem_used={mem_used_mb:.0f}MB/{mem_total_mb:.0f}MB"
    except Exception:
        pass

    if DEVICE == 'cuda' and torch.cuda.is_available():
        free_b, total_b = torch.cuda.mem_get_info()
        runtime_used_mb = (total_b - free_b) / (1024.0 ** 2)
        runtime_total_mb = total_b / (1024.0 ** 2)
        allocated_mb = torch.cuda.memory_allocated() / (1024.0 ** 2)
        reserved_mb = torch.cuda.memory_reserved() / (1024.0 ** 2)
        print(
            f"epoch={epoch_num} memory: {sys_msg}; "
            f"gpu_runtime_used={runtime_used_mb:.0f}MB/{runtime_total_mb:.0f}MB; "
            f"gpu_allocated={allocated_mb:.0f}MB; gpu_reserved={reserved_mb:.0f}MB"
        )
    else:
        print(f"epoch={epoch_num} memory: {sys_msg}; gpu_mem=not_applicable")

def _resolve_resume_checkpoint_path(output_dir: str, configured_path: Optional[str]) -> Optional[str]:
    candidates = []
    if configured_path:
        candidates.append(configured_path)
    candidates.append(os.path.join(output_dir, "last_checkpoint.pt"))
    candidates.append(os.path.join(output_dir, "best_checkpoint.pt"))
    candidates.append(os.path.join(output_dir, "best_model.pt"))  # weights-only fallback

    for path in candidates:
        if path and os.path.exists(path):
            return path
    return None


def _estimate_true_none_prevalence(dataset: SpanDataset) -> float:
    """Estimate NONE prevalence without negative subsampling."""
    none_count = 0
    total_count = 0

    for idx, row in enumerate(dataset.rows):
        text = row["text"]
        gold_spans = row.get("spans", [])

        enc = dataset.tokenizer(
            text,
            truncation=True,
            max_length=dataset.max_length,
            return_offsets_mapping=True,
            return_attention_mask=True,
        )
        offset_mapping = enc["offset_mapping"]
        valid_token_indices = [i for i, (s, e) in enumerate(offset_mapping) if e > s]

        num_valid_tokens = len(valid_token_indices)
        if num_valid_tokens == 0:
            continue

        # Count all span candidates of length 1..max_span_len without explicit enumeration.
        total_candidates = 0
        max_len_here = min(dataset.max_span_len, num_valid_tokens)
        for span_len in range(1, max_len_here + 1):
            total_candidates += num_valid_tokens - span_len + 1

        span_label_map = {}
        for gs in gold_spans:
            mapped = char_span_to_token_span(offset_mapping, gs["start"], gs["end"])
            if mapped is None:
                continue

            raw_label = gs["label"]
            if raw_label not in dataset.label2id:
                raise ValueError(
                    f"Unknown span label '{raw_label}' in dataset row {idx}. "
                    f"Expected one of: {sorted(dataset.label2id.keys())}. "
                    "Update ENTITY_LABELS or normalize your dataset labels."
                )
            span_label_map[mapped] = dataset.label2id[raw_label]

        positive_count = len(span_label_map)
        none_count += max(total_candidates - positive_count, 0)
        total_count += total_candidates

    if total_count == 0:
        return 1.0
    return none_count / total_count


def _target_none_prevalence_for_epoch(
    epoch_idx: int,
    true_none_prevalence: float,
    start_none_prevalence: float = 0.10,
    full_prevalence_epoch: int = 30,
) -> float:
    if full_prevalence_epoch <= 0:
        return float(true_none_prevalence)

    progress = min(max(epoch_idx, 0), full_prevalence_epoch) / float(full_prevalence_epoch)
    return float(start_none_prevalence + (true_none_prevalence - start_none_prevalence) * progress)


def _negative_subsample_for_target_none(
    target_none_prevalence: float,
    true_none_prevalence: float,
) -> float:
    """
    Solve p in: target = (p * true_none) / (p * true_none + (1 - true_none)).
    p is the keep probability for NONE spans.
    """
    if true_none_prevalence <= 0.0:
        return 1.0
    if true_none_prevalence >= 1.0:
        return 1.0

    t = min(max(target_none_prevalence, 0.0), 1.0 - 1e-9)
    n = true_none_prevalence
    p = (t * (1.0 - n)) / (n * (1.0 - t))
    return float(min(max(p, 0.0), 1.0))


def _partial_overlap_weight_for_epoch(
    epoch_idx: int,
    start_epoch: int,
    full_epoch: int,
    max_weight: float,
) -> float:
    if epoch_idx < start_epoch:
        return 0.0
    if full_epoch <= start_epoch:
        return float(max_weight)

    progress = (epoch_idx - start_epoch) / float(full_epoch - start_epoch)
    progress = min(max(progress, 0.0), 1.0)
    return float(max_weight * progress)


def _estimate_reachable_span_stats(dataset: SpanDataset):
    """Estimate share of gold spans reachable under dataset.max_span_len (token spans)."""
    totals = Counter()
    reachable = Counter()

    for idx, row in enumerate(dataset.rows):
        text = row["text"]
        gold_spans = row.get("spans", [])

        enc = dataset.tokenizer(
            text,
            truncation=True,
            max_length=dataset.max_length,
            return_offsets_mapping=True,
            return_attention_mask=True,
        )
        offset_mapping = enc["offset_mapping"]

        for gs in gold_spans:
            raw_label = gs.get("label")
            if raw_label not in dataset.label2id:
                continue
            mapped = char_span_to_token_span(offset_mapping, gs["start"], gs["end"])
            if mapped is None:
                continue
            totals[raw_label] += 1
            span_len = mapped[1] - mapped[0] + 1
            if span_len <= dataset.max_span_len:
                reachable[raw_label] += 1

    total_gold = sum(totals.values())
    total_reachable = sum(reachable.values())
    per_label = {}
    for lbl in [l for l in LABELS if l != "NONE"]:
        t = int(totals.get(lbl, 0))
        r = int(reachable.get(lbl, 0))
        per_label[lbl] = {
            "total": t,
            "reachable": r,
            "reachable_rate": (r / t) if t > 0 else 0.0,
        }

    return {
        "total_gold": int(total_gold),
        "total_reachable": int(total_reachable),
        "reachable_rate": (total_reachable / total_gold) if total_gold > 0 else 0.0,
        "per_label": per_label,
    }


def _log_calibration_report(
    epoch_num: int,
    prev_thresholds: Optional[Dict[str, float]],
    new_thresholds: Dict[str, float],
    calibration_report: Dict[str, dict],
    top_k: int = 8,
):
    delta_rows = []
    for lbl, thr in new_thresholds.items():
        old = prev_thresholds.get(lbl, EVAL_THRESHOLD) if prev_thresholds else EVAL_THRESHOLD
        delta_rows.append((abs(float(thr) - float(old)), lbl, float(old), float(thr)))
    delta_rows.sort(reverse=True)

    changed = [row for row in delta_rows if row[0] > 1e-6]
    print(f"epoch={epoch_num} calibration changed_labels={len(changed)}/{len(new_thresholds)}")

    if changed:
        for _, lbl, old, new in changed[: max(top_k, 1)]:
            rep = calibration_report.get(lbl, {})
            support = rep.get("support", 0)
            examples = rep.get("examples", 0)
            best_f1 = rep.get("best_f1_surrogate", 0.0)
            print(
                f"  calib[{lbl}] {old:.4f}->{new:.4f}; "
                f"support={support}, argmax_examples={examples}, f1_sur={best_f1:.4f}"
            )

    fallback_labels = [lbl for lbl, rep in calibration_report.items() if rep.get("default_used", False)]
    if fallback_labels:
        preview = ", ".join(fallback_labels[: max(top_k, 1)])
        more = "" if len(fallback_labels) <= top_k else f" (+{len(fallback_labels)-top_k} more)"
        print(f"  calib default-threshold labels: {preview}{more}")


def _log_metric_diagnostics(epoch_num: int, split_name: str, metrics: dict):
    tp = int(metrics.get("tp", 0))
    fp = int(metrics.get("fp", 0))
    fn = int(metrics.get("fn", 0))
    wrong_place = int(metrics.get("wrong_place_count", 0))
    wrong_overlap = int(metrics.get("wrong_place_overlap_count", 0))

    overlap_share = (wrong_overlap / wrong_place) if wrong_place > 0 else 0.0
    fp_per_tp = (fp / tp) if tp > 0 else float("inf")
    fn_per_tp = (fn / tp) if tp > 0 else float("inf")

    decoded_counts = metrics.get("pred_decoded_label_counts", {})
    decoded_top = sorted(decoded_counts.items(), key=lambda x: x[1], reverse=True)[:5]

    print(
        f"epoch={epoch_num} {split_name} diagnostics: "
        f"fp/tp={fp_per_tp:.2f}, fn/tp={fn_per_tp:.2f}, overlap_share={overlap_share:.3f}"
    )
    print(f"epoch={epoch_num} {split_name} top decoded labels: {decoded_top}")

    dead_labels = []
    for lbl, stat in metrics.get("per_label_metrics", {}).items():
        support = int(stat.get("support", 0))
        lbl_tp = int(stat.get("tp", 0))
        if support >= LOG_DEAD_LABELS_MIN_SUPPORT and lbl_tp == 0:
            dead_labels.append(lbl)
    if dead_labels:
        print(f"epoch={epoch_num} {split_name} zero-tp labels (support>={LOG_DEAD_LABELS_MIN_SUPPORT}): {dead_labels}")


true_none_prevalence = _estimate_true_none_prevalence(train_ds)
print(f"[none schedule] True NONE prevalence in train candidates: {true_none_prevalence:.6f}")

if LOG_REACHABILITY_SUMMARY:
    train_reach = _estimate_reachable_span_stats(train_ds)
    valid_reach = _estimate_reachable_span_stats(valid_ds)
    print(
        f"[reachability] train reachable={train_reach['total_reachable']}/{train_reach['total_gold']} "
        f"({train_reach['reachable_rate']:.3f}) at max_span_len={train_ds.max_span_len}"
    )
    print(
        f"[reachability] valid reachable={valid_reach['total_reachable']}/{valid_reach['total_gold']} "
        f"({valid_reach['reachable_rate']:.3f}) at max_span_len={valid_ds.max_span_len}"
    )
    worst_labels = sorted(
        valid_reach["per_label"].items(),
        key=lambda kv: kv[1]["reachable_rate"]
    )[:5]
    print(
        "[reachability] valid worst labels: "
        + str([(lbl, round(stats["reachable_rate"], 3), stats["reachable"], stats["total"]) for lbl, stats in worst_labels])
    )

best_f1 = -1.0
best_epoch = 0
epochs_without_improve = 0
start_epoch = 0
current_label_thresholds = None
os.makedirs(OUTPUT_DIR, exist_ok=True)

if RESUME_TRAINING:
    resume_path = _resolve_resume_checkpoint_path(OUTPUT_DIR, RESUME_CHECKPOINT_PATH)
    if resume_path:
        print(f"[resume] Loading checkpoint: {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)

        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"])
        elif isinstance(ckpt, dict) and "model" in ckpt:
            model.load_state_dict(ckpt["model"])
        else:
            model.load_state_dict(ckpt)

        if isinstance(ckpt, dict) and "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if isinstance(ckpt, dict) and "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        if isinstance(ckpt, dict):
            start_epoch = int(ckpt.get("epoch", 0))
            best_f1 = float(ckpt.get("best_f1", best_f1))
            best_epoch = int(ckpt.get("best_epoch", best_epoch))
            epochs_without_improve = int(ckpt.get("epochs_without_improve", epochs_without_improve))
            current_label_thresholds = ckpt.get("label_thresholds", current_label_thresholds)

        print(
            f"[resume] start_epoch={start_epoch}, best_epoch={best_epoch}, "
            f"best_f1={best_f1:.6f}, no_improve={epochs_without_improve}, "
            f"label_thresholds_loaded={current_label_thresholds is not None}"
        )
    else:
        print("[resume] No checkpoint found; starting fresh training.")

for epoch in range(start_epoch, NUM_EPOCHS):
    if DEVICE == "cuda" and torch.cuda.is_available() and LOG_MEMORY_EVERY_EPOCH:
        torch.cuda.reset_peak_memory_stats()

    target_none_prevalence = _target_none_prevalence_for_epoch(
        epoch_idx=epoch,
        true_none_prevalence=true_none_prevalence,
        start_none_prevalence=NONE_START_PREVALENCE,
        full_prevalence_epoch=NONE_FULL_PREVALENCE_EPOCH,
    )
    train_ds.negative_span_subsample = _negative_subsample_for_target_none(
        target_none_prevalence=target_none_prevalence,
        true_none_prevalence=true_none_prevalence,
    )
    train_ds.partial_overlap_weight = _partial_overlap_weight_for_epoch(
        epoch_idx=epoch,
        start_epoch=PARTIAL_OVERLAP_START_EPOCH,
        full_epoch=PARTIAL_OVERLAP_FULL_EPOCH,
        max_weight=PARTIAL_OVERLAP_MAX_WEIGHT,
    )
    print(
        f"epoch={epoch+1} none schedule: "
        f"target_none={target_none_prevalence:.6f}, "
        f"true_none={true_none_prevalence:.6f}, "
        f"negative_subsample={train_ds.negative_span_subsample:.6f}, "
        f"partial_overlap_weight={train_ds.partial_overlap_weight:.4f}"
    )

    model.train()
    optimizer.zero_grad()
    running_loss = 0.0

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"train epoch {epoch+1}")
    for step, batch in pbar:
        input_ids = batch.input_ids.to(DEVICE)
        attention_mask = batch.attention_mask.to(DEVICE)
        spans = batch.spans.to(DEVICE)
        span_batch_ix = batch.span_batch_ix.to(DEVICE)
        labels = batch.labels.to(DEVICE)
        loss_weights = batch.loss_weights.to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            spans=spans,
            span_batch_ix=span_batch_ix,
            labels=labels,
            loss_weights=loss_weights,
        )
        loss = outputs["loss"] / GRAD_ACCUM_STEPS
        loss.backward()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        pbar.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = running_loss / max(len(train_loader), 1)

    use_label_thresholds = USE_PER_LABEL_THRESHOLDING and ((epoch + 1) > PER_LABEL_THRESHOLD_WARMUP_EPOCHS)

    should_calibrate = (
        use_label_thresholds
        and (
            current_label_thresholds is None
            or ((epoch + 1) % max(PER_LABEL_CALIBRATION_EVERY_N_EPOCHS, 1) == 0)
        )
    )
    if should_calibrate:
        previous_thresholds = dict(current_label_thresholds) if current_label_thresholds is not None else None
        current_label_thresholds, calibration_report = fit_label_thresholds_from_argmax(
            model,
            valid_loader,
            default_threshold=EVAL_THRESHOLD,
            min_support=PER_LABEL_CALIBRATION_MIN_SUPPORT,
        )
        _log_calibration_report(
            epoch_num=epoch + 1,
            prev_thresholds=previous_thresholds,
            new_thresholds=current_label_thresholds,
            calibration_report=calibration_report,
            top_k=LOG_CALIBRATION_TOP_K,
        )

    active_label_thresholds = current_label_thresholds if use_label_thresholds else None

    train_metrics = evaluate_model(
        model,
        train_loader,
        threshold=EVAL_THRESHOLD,
        label_thresholds=active_label_thresholds,
    )
    valid_metrics = evaluate_model(
        model,
        valid_loader,
        threshold=EVAL_THRESHOLD,
        label_thresholds=active_label_thresholds,
    )

    compact_train = {
        "precision": float(train_metrics.get("precision", 0.0)),
        "recall": float(train_metrics.get("recall", 0.0)),
        "f1": float(train_metrics.get("f1", 0.0)),
        "tp": int(train_metrics.get("tp", 0)),
        "fp": int(train_metrics.get("fp", 0)),
        "fn": int(train_metrics.get("fn", 0)),
    }
    compact_valid = {
        "precision": float(valid_metrics.get("precision", 0.0)),
        "recall": float(valid_metrics.get("recall", 0.0)),
        "f1": float(valid_metrics.get("f1", 0.0)),
        "tp": int(valid_metrics.get("tp", 0)),
        "fp": int(valid_metrics.get("fp", 0)),
        "fn": int(valid_metrics.get("fn", 0)),
    }

    print(f"epoch={epoch+1} train loss: {avg_train_loss:.6f}")
    if use_label_thresholds and current_label_thresholds is not None:
        print(f"epoch={epoch+1} monitor threshold: per-label + fallback={EVAL_THRESHOLD}")
    elif USE_PER_LABEL_THRESHOLDING and (epoch + 1) <= PER_LABEL_THRESHOLD_WARMUP_EPOCHS:
        print(
            f"epoch={epoch+1} monitor threshold: global={EVAL_THRESHOLD} "
            f"(per-label warmup epoch {epoch+1}/{PER_LABEL_THRESHOLD_WARMUP_EPOCHS})"
        )
    else:
        print(f"epoch={epoch+1} monitor threshold: {EVAL_THRESHOLD}")
    print(f"epoch={epoch+1} train metrics (compact): {compact_train}")
    print(f"epoch={epoch+1} valid metrics (compact): {compact_valid}")
    _log_metric_diagnostics(epoch + 1, "train", train_metrics)
    _log_metric_diagnostics(epoch + 1, "valid", valid_metrics)

    improved = valid_metrics["f1"] > (best_f1 + MIN_DELTA)
    if improved:
        best_f1 = valid_metrics["f1"]
        best_epoch = epoch + 1
        epochs_without_improve = 0

        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pt"))

        best_ckpt = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "epochs_without_improve": epochs_without_improve,
            "label_thresholds": current_label_thresholds,
        }
        torch.save(best_ckpt, os.path.join(OUTPUT_DIR, "best_checkpoint.pt"))

        tokenizer.save_pretrained(OUTPUT_DIR)
        with open(os.path.join(OUTPUT_DIR, "labels.json"), "w", encoding="utf-8") as f:
            json.dump({"labels": LABELS}, f, ensure_ascii=False, indent=2)
        with open(os.path.join(OUTPUT_DIR, "label_thresholds.json"), "w", encoding="utf-8") as f:
            json.dump({
                "fallback_threshold": EVAL_THRESHOLD,
                "label_thresholds": current_label_thresholds,
            }, f, ensure_ascii=False, indent=2)
        print("saved best model + best checkpoint + label thresholds")
    else:
        epochs_without_improve += 1

    if LOG_FULL_METRICS_EVERY_N_EPOCHS > 0 and ((epoch + 1) % LOG_FULL_METRICS_EVERY_N_EPOCHS == 0 or improved):
        print(f"epoch={epoch+1} train metrics (full):", train_metrics)
        print(f"epoch={epoch+1} valid metrics (full):", valid_metrics)

    last_ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "epochs_without_improve": epochs_without_improve,
        "label_thresholds": current_label_thresholds,
    }
    torch.save(last_ckpt, os.path.join(OUTPUT_DIR, "last_checkpoint.pt"))

    if SAVE_EVERY_N_EPOCHS > 0 and (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        periodic_ckpt_path = os.path.join(OUTPUT_DIR, f"epoch_{epoch+1:04d}_checkpoint.pt")
        torch.save(last_ckpt, periodic_ckpt_path)
        print(f"saved periodic checkpoint: {periodic_ckpt_path}")

    if LOG_MEMORY_EVERY_EPOCH:
        _log_memory(epoch + 1)

    if ENABLE_MEMORY_HOUSEKEEPING:
        del train_metrics
        del valid_metrics
        gc_module = globals().get("gc")
        if gc_module is not None:
            gc_module.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    print(
        f"early-stop monitor: no_improve={epochs_without_improve}/{EARLY_STOPPING_PATIENCE}, "
        f"best_epoch={best_epoch}, best_f1={best_f1:.6f}"
    )

    if (epoch + 1) >= MIN_EPOCHS and epochs_without_improve >= EARLY_STOPPING_PATIENCE:
        print(
            f"early stopping at epoch {epoch+1} "
            f"(best epoch={best_epoch}, best_f1={best_f1:.6f}, "
            f"no improvement for {EARLY_STOPPING_PATIENCE} epochs)"
        )
        break


[none schedule] True NONE prevalence in train candidates: 0.996875
[reachability] train reachable=10085/10129 (0.996) at max_span_len=15
[reachability] valid reachable=2066/2066 (1.000) at max_span_len=15
[reachability] valid worst labels: [('PERSON', 1.0, 255, 255), ('ORG', 1.0, 253, 253), ('ADDRESS', 1.0, 274, 274), ('EMAIL', 1.0, 291, 291), ('PHONE', 1.0, 218, 218)]
epoch=1 none schedule: target_none=0.100000, true_none=0.996875, negative_subsample=0.000348, partial_overlap_weight=0.0000


train epoch 1:  61%|██████    | 217/357 [00:31<00:20,  6.90it/s, loss=2.6534]


KeyboardInterrupt: 

In [ ]:
# -----------------------------
# Load best model and run a tiny demo
# -----------------------------
best_model = ModernBertForSpanClassification(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    max_span_len=MAX_SPAN_LEN,
    length_embed_dim=LENGTH_EMBED_DIM,
    dropout=DROPOUT,
).to(DEVICE)

best_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE))
best_model.eval()

inference_label_thresholds = None
label_thresholds_path = os.path.join(OUTPUT_DIR, "label_thresholds.json")
if os.path.exists(label_thresholds_path):
    with open(label_thresholds_path, "r", encoding="utf-8") as f:
        loaded = json.load(f)
        inference_label_thresholds = loaded.get("label_thresholds")

def predict_spans(text: str, threshold: float = 0.5):
    row = {"text": text, "spans": []}
    tmp_path = "_tmp_predict.jsonl"
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    ds = SpanDataset(
        tmp_path,
        tokenizer=tokenizer,
        label2id=label2id,
        max_length=MAX_LENGTH,
        max_span_len=MAX_SPAN_LEN,
        negative_span_subsample=1.0,
        training=False,
    )
    batch = collate_fn([ds[0]])

    with torch.no_grad():
        outputs = best_model(
            input_ids=batch.input_ids.to(DEVICE),
            attention_mask=batch.attention_mask.to(DEVICE),
            spans=batch.spans.to(DEVICE),
            span_batch_ix=batch.span_batch_ix.to(DEVICE),
            labels=None,
        )

    resolved = decode_predictions(
        batch,
        outputs["logits"].cpu(),
        threshold=threshold,
        label_thresholds=inference_label_thresholds,
    )[0]
    pred_char = token_spans_to_char_spans(resolved, batch.meta[0]["offset_mapping"])

    results = []
    for s, e, lbl, score in pred_char:
        results.append({
            "text": text[s:e],
            "start": s,
            "end": e,
            "label": lbl,
            "score": round(float(score), 4),
        })
    return results

demo_text = "John Smith lives at Bahnhofstrasse 1 in Zurich and works for Google."
predict_spans(demo_text, threshold=0)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 13672.53it/s]
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'John Smith',
  'start': 0,
  'end': 10,
  'label': 'PERSON',
  'score': 0.5385}]

In [ ]:
# -----------------------------
# Validation diagnostics: print 50 examples with overlap signals
# -----------------------------

def _char_overlap(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    # End indices are exclusive in this dataset.
    return max(a_start, b_start) < min(a_end, b_end)


def _top_span_predictions_for_item(batch: Batch, logits: torch.Tensor, item_idx: int, top_k: int = 10):
    probs = torch.softmax(logits, dim=-1)
    none_id = label2id["NONE"]

    candidates = []
    for span_idx in range(len(batch.spans)):
        if int(batch.span_batch_ix[span_idx]) != item_idx:
            continue

        tok_start, tok_end = batch.spans[span_idx].tolist()
        char_start = batch.meta[item_idx]["offset_mapping"][tok_start][0]
        char_end = batch.meta[item_idx]["offset_mapping"][tok_end][1]

        span_probs = probs[span_idx].clone()
        span_probs[none_id] = -1.0
        pred_label_id = int(span_probs.argmax().item())
        pred_score = float(probs[span_idx][pred_label_id].item())
        none_score = float(probs[span_idx][none_id].item())

        candidates.append({
            "start": char_start,
            "end": char_end,
            "label": id2label[pred_label_id],
            "score": pred_score,
            "none_score": none_score,
        })

    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates[:top_k]


def print_validation_examples_with_overlaps(model, dataset, num_examples: int = 50, top_k: int = 10):
    model.eval()
    total = min(num_examples, len(dataset))

    for idx in range(total):
        item = dataset[idx]
        batch = collate_fn([item])

        with torch.no_grad():
            outputs = model(
                input_ids=batch.input_ids.to(DEVICE),
                attention_mask=batch.attention_mask.to(DEVICE),
                spans=batch.spans.to(DEVICE),
                span_batch_ix=batch.span_batch_ix.to(DEVICE),
                labels=None,
            )

        top_preds = _top_span_predictions_for_item(batch, outputs["logits"].cpu(), item_idx=0, top_k=top_k)

        text = batch.meta[0]["text"]
        gold_spans = [
            {
                "start": int(gs["start"]),
                "end": int(gs["end"]),
                "label": gs["label"],
            }
            for gs in batch.meta[0]["gold_spans"]
        ]

        print("=" * 120)
        print(f"Example {idx + 1}/{total}")
        print(f"Text: {text}")

        print("Gold spans:")
        if not gold_spans:
            print("  - <none>")
        else:
            for gs in gold_spans:
                snippet = text[gs['start']:gs['end']]
                print(
                    f"  - [{gs['label']}] ({gs['start']}, {gs['end']}) "
                    f"'{snippet}'"
                )

        print(f"Top predicted spans (top_k={top_k}):")
        if not top_preds:
            print("  - <none>")
            continue

        for rank, pred in enumerate(top_preds, start=1):
            overlaps = []
            exact_match = False
            same_label_overlap = False

            for gs in gold_spans:
                if (
                    pred["start"] == gs["start"]
                    and pred["end"] == gs["end"]
                    and pred["label"] == gs["label"]
                ):
                    exact_match = True

                if _char_overlap(pred["start"], pred["end"], gs["start"], gs["end"]):
                    overlaps.append(gs["label"])
                    if pred["label"] == gs["label"]:
                        same_label_overlap = True

            if exact_match:
                overlap_indicator = "EXACT"
            elif same_label_overlap:
                overlap_indicator = "OVERLAP_SAME_LABEL"
            elif overlaps:
                overlap_indicator = "OVERLAP_DIFF_LABEL"
            else:
                overlap_indicator = "NO_OVERLAP"

            snippet = text[pred['start']:pred['end']]
            overlap_labels = sorted(set(overlaps))
            overlap_labels_str = ",".join(overlap_labels) if overlap_labels else "-"

            print(
                f"  {rank:02d}. [{pred['label']}] score={pred['score']:.4f} "
                f"none={pred['none_score']:.4f} ({pred['start']}, {pred['end']}) "
                f"'{snippet}' -> {overlap_indicator}; overlap_labels={overlap_labels_str}"
            )


print_validation_examples_with_overlaps(best_model, valid_ds, num_examples=50, top_k=10)


Example 1/50
Text: John Smith lives at Bahnhofstrasse 12, Zürich. Email him at john.smith@example.com.
Gold spans:
  - [PERSON] (0, 10) 'John Smith'
  - [ADDRESS] (20, 46) 'Bahnhofstrasse 12, Zürich.'
  - [EMAIL] (61, 84) 'ohn.smith@example.com.'
Top predicted spans (top_k=10):
  01. [PERSON] score=0.9952 none=0.0047 (0, 10) 'John Smith' -> EXACT; overlap_labels=PERSON
  02. [PERSON] score=0.0353 none=0.9645 (0, 16) 'John Smith lives' -> OVERLAP_SAME_LABEL; overlap_labels=PERSON
  03. [PERSON] score=0.0308 none=0.9690 (0, 21) 'John Smith lives at B' -> OVERLAP_SAME_LABEL; overlap_labels=ADDRESS,PERSON
  04. [PERSON] score=0.0099 none=0.9899 (0, 4) 'John' -> OVERLAP_SAME_LABEL; overlap_labels=PERSON
  05. [PERSON] score=0.0013 none=0.9987 (0, 24) 'John Smith lives at Bahn' -> OVERLAP_SAME_LABEL; overlap_labels=ADDRESS,PERSON
  06. [PERSON] score=0.0003 none=0.9997 (0, 19) 'John Smith lives at' -> OVERLAP_SAME_LABEL; overlap_labels=PERSON
  07. [PASSWORD] score=0.0001 none=0.9997 (19, 37

## Where to improve next

This is a baseline, not a cathedral.

The most useful upgrades are usually:

1. **Better negatives**
   - downsample `NONE`
   - add hard negatives that resemble names, IDs, addresses

2. **Better decoding**
   - tune thresholds per label
   - use label-specific overlap rules

3. **More efficient span pooling**
   - replace the Python loop over spans with prefix-sum pooling or batched gather ops

4. **Longer labels**
   - keep `MAX_SPAN_LEN=6` for names/orgs
   - handle longer addresses with a separate component or larger cap

5. **Metrics**
   - compute per-label span F1
   - track document-level leakage

6. **Structured PII**
   - let regex/validators handle emails, phones, credit cards, IBANs, IPs
   - let the span classifier focus on fuzzy things like names and free-form addresses
